# Z24 experiment runner

This notebook contains no model logic. Select one config, then run all cells. Attach the dataset containing `inputs.npy` and `labels.npy`; the runner locates it under `/kaggle/input`. Run each experiment in a fresh Kaggle session so TensorFlow and PyTorch do not retain each other's GPU memory.

In [ ]:
# Cell 1 - settings you may change
from pathlib import Path

REPO_URL = 'https://github.com/tranvanphuongdevdream-web/shm_ml.git'
BRANCH = 'master'
EXPERIMENT_ID = 'dcnn_002'  # dcnn_001 | dcnn_002 | tsai_001
OUTPUT_DIR = Path('/kaggle/working/results')
REPO_DIR = Path('/kaggle/working/shm_ml')

assert EXPERIMENT_ID in {'dcnn_001', 'dcnn_002', 'tsai_001'}


In [ ]:
# Cell 2 - clone once; reuse the local clone during this Kaggle session
import importlib
import os
import subprocess
import sys

git_environment = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def run_git(arguments):
    return subprocess.run(
        arguments, check=True, timeout=180, env=git_environment,
    )

if (REPO_DIR / '.git').is_dir():
    print('Reusing repository already cloned in this session.', flush=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
else:
    print('Cloning repository...', flush=True)
    run_git(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])

commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True,
).strip()
print(f'Repository ready: {REPO_DIR} (commit {commit})', flush=True)
sys.path.insert(0, str(REPO_DIR))
from src.data import z24_dataset
importlib.reload(z24_dataset)  # use code just pulled, not a cached module
INPUTS_PATH, LABELS_PATH = z24_dataset.resolve_data_source()
print('Dataset inputs:', INPUTS_PATH)
print('Dataset labels:', LABELS_PATH)


In [ ]:
# Cell 3 - install only the dependencies needed by the selected experiment
import hashlib
import importlib
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement

def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

if EXPERIMENT_ID == 'tsai_001':
    tsai_package_names = {
        'scikit-learn', 'fastai', 'imbalanced-learn', 'pyts', 'psutil', 'tsai',
    }
    parsed = [
        Requirement(line.strip())
        for line in (REPO_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    ]
    selected = [item for item in parsed if item.name.lower() in tsai_package_names]
    tsai_items = [item for item in selected if item.name.lower() == 'tsai']
    if len(tsai_items) != 1:
        raise ValueError('requirements.txt must contain exactly one tsai requirement')
    tsai_item = tsai_items[0]
    other_items = [item for item in selected if item is not tsai_item]
    wheel_dir = REPO_DIR / 'vendor' / 'wheels'
    expected_hashes = {
        filename: digest
        for digest, filename in (
            line.split(maxsplit=1)
            for line in (wheel_dir / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines()
            if line.strip()
        )
    }

    def bundled_wheel(item):
        prefix = item.name.lower().replace('-', '_') + '-'
        matches = sorted(
            path for path in wheel_dir.glob('*.whl')
            if path.name.lower().startswith(prefix)
        )
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected one bundled wheel for {item.name} in {wheel_dir}, found {matches}'
            )
        wheel = matches[0]
        actual_hash = hashlib.sha256(wheel.read_bytes()).hexdigest()
        expected_hash = expected_hashes.get(wheel.name)
        if actual_hash != expected_hash:
            raise ValueError(f'SHA-256 mismatch for bundled wheel: {wheel}')
        return wheel

    def install_bundled(item):
        wheel = bundled_wheel(item)
        print(f'Installing offline wheel: {wheel.name}', flush=True)
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--no-index',
            '--no-deps', str(wheel),
        ], check=True, timeout=180)

    missing_items = [
        item for item in other_items if installed_version(item.name) is None
    ]
    torch_before = installed_version('torch')
    print('Using Kaggle preinstalled packages:', flush=True)
    for item in other_items:
        current = installed_version(item.name)
        if current is not None:
            print(f'  {item.name}=={current}', flush=True)
    if missing_items:
        print('Installing missing packages from bundled wheels...', flush=True)
        for position, item in enumerate(missing_items, start=1):
            print(f'[{position}/{len(missing_items)}] Installing {item}', flush=True)
            install_bundled(item)
    tsai_installed = installed_version('tsai')
    if tsai_installed is None or not tsai_item.specifier.contains(tsai_installed):
        print(f'Installing {tsai_item} without replacing PyTorch...', flush=True)
        install_bundled(tsai_item)
    importlib.invalidate_caches()
    unresolved = [item.name for item in selected if installed_version(item.name) is None]
    if unresolved:
        raise RuntimeError(f'Missing dependencies after installation: {unresolved}')
    if installed_version('torch') != torch_before:
        raise RuntimeError(
            f'PyTorch changed unexpectedly: {torch_before} -> {installed_version("torch")}. '
            'Restart the Kaggle session before continuing.'
        )
    module_names = {
        'scikit-learn': 'sklearn', 'imbalanced-learn': 'imblearn',
    }
    for item in selected:
        importlib.import_module(module_names.get(item.name.lower(), item.name.lower()))
    print('Validated tsai imports successfully.', flush=True)
print('Dependencies ready for:', EXPERIMENT_ID, flush=True)


In [ ]:
# Cell 4 - run training directly so Kaggle displays every progress log
import time
from datetime import datetime

existing_runs = {path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*') if path.is_dir()}
started_at = time.perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Starting training', flush=True)
print(f'Experiment: {EXPERIMENT_ID}', flush=True)
print('Live progress will appear below. The first GPU graph compilation may take a few minutes.', flush=True)
from src import run_experiment
importlib.reload(run_experiment)
run_experiment.run('train', EXPERIMENT_ID)
elapsed_minutes = (time.perf_counter() - started_at) / 60
print(f'[{datetime.now():%H:%M:%S}] Training finished in {elapsed_minutes:.2f} minutes', flush=True)
new_runs = [
    path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*')
    if path.is_dir() and path not in existing_runs
]
if len(new_runs) != 1:
    raise RuntimeError(f'Expected one new result directory, found: {new_runs}')
RUN_DIR = new_runs[0]
print('Results:', RUN_DIR)
print('Download ZIP:', RUN_DIR.with_suffix('.zip'))


In [ ]:
# Cell 5 - metrics and training benchmark for this run
import json
import pandas as pd
from IPython.display import display

metrics = pd.read_csv(RUN_DIR / 'split_metrics.csv', index_col='split')
benchmark = json.loads((RUN_DIR / 'benchmark_summary.json').read_text(encoding='utf-8'))
print('Accuracy, macro precision, macro recall, and macro F1:')
display(metrics.style.format('{:.2%}'))

benchmark_rows = [
    ('Experiment', benchmark['pipeline']),
    ('Total training time (min)', round(benchmark['training_seconds_total'] / 60, 2)),
    ('Epochs completed', benchmark['epochs_completed']),
    ('Mean epoch time (s)', round(benchmark['mean_epoch_seconds'], 2)),
    ('Mean epoch time, excluding first (s)', round(benchmark['mean_epoch_seconds_excluding_first'], 2)),
    ('Train samples per second', round(benchmark['effective_train_samples_per_second'], 2)),
    ('Model parameters', benchmark['model_parameters']),
    ('Batch size', benchmark['batch_size']),
    ('GPU', ', '.join(benchmark['gpu_names']) or 'CPU'),
    ('Precision policy', benchmark['precision_policy']),
]
print('Training benchmark:')
display(pd.DataFrame(benchmark_rows, columns=['Metric', 'Value']).set_index('Metric'))


In [ ]:
# Cell 6 - learning curves, split comparison, and test confusion matrix
import matplotlib.pyplot as plt
from IPython.display import Image

history = pd.read_csv(RUN_DIR / 'history.csv')
if not history.empty:
    epoch = history['epoch'] if 'epoch' in history else range(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for axis, title, columns in (
        (axes[0], 'Loss by epoch', ('loss', 'val_loss', 'train_loss', 'valid_loss')),
        (axes[1], 'Accuracy by epoch', ('accuracy', 'val_accuracy')),
    ):
        available = [column for column in columns if column in history]
        for column in available:
            axis.plot(epoch, history[column], label=column)
        if available:
            axis.set_title(title)
            axis.set_xlabel('Epoch')
            axis.grid(True, alpha=0.3)
            axis.legend()
        else:
            axis.set_visible(False)
    fig.tight_layout()
    plt.show()

axis = metrics[['accuracy', 'f1_macro']].plot.bar(figsize=(8, 4), rot=0)
axis.set_title('Performance by split')
axis.set_ylabel('Score')
axis.set_ylim(0, 1)
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Test confusion matrix:')
display(Image(filename=str(RUN_DIR / 'test_confusion_matrix.png'), width=850))
